# Train a YOLO classifier

YOLO (You Only Look Once) is a popular neural-network-based algorithm for image classification, object detection and segmentation.

Here we're leveraging a pre-trained YOLO model for image classification and re-training its last layers to make it perform a more specific classification task.

In order to achieve this we need to give the YOLO trainer module a set of images to train on, and a separate set of images to evaluate its performance.

To set this up in the way that YOLO expects the images, we need to have a directory with the name of your dataset, and inside that directory, two directories named `train/` and `test/`. And then, each of these two directories would have a directory for each of the classes that you're trying to classify.

For example, if I want to classify pictures of constellations I would have the following directories with training data:

- `stars/train/orion`
- `stars/train/ursa-major`
- `stars/train/ursa-minor`
- `stars/train/cassiopeia`
- `stars/train/cygnus`
- `stars/train/scorpius`

Each of these would have $20$ to $30$ images of each constellation, ideally with the class name in the filename (`00_orion.jpg`, `100_cygnus.png`, etc.)

I would also have these directories with test data ($10$ to $15$ images of each constellation):

- `stars/test/orion`
- `stars/test/ursa-major`
- `stars/test/ursa-minor`
- `stars/test/cassiopeia`
- `stars/test/cygnus`
- `stars/test/scorpius`


### Install the YOLO library

In [4]:
!pip install ultralytics

In [5]:
from pathlib import Path
from urllib.request import urlretrieve

YOLO_UTILS_URL = "https://github.com/PSAM-5005-2026S-A/5005-utils/raw/main/src/yolo_utils.py"
YOLO_UTILS_PATH = Path("yolo_utils.py")

if not YOLO_UTILS_PATH.exists():
    try:
        urlretrieve(YOLO_UTILS_URL, YOLO_UTILS_PATH)
        print(f"Downloaded: {YOLO_UTILS_PATH.resolve()}")
    except Exception as exc:
        print(f"Could not download yolo_utils.py: {exc}")

print(f"Helper file exists: {YOLO_UTILS_PATH.exists()}")

Downloaded: /Users/aishakazembe/Documents/coding-cabinate/code/Intro_to_data/Intro_to_data/5005-Project02/yolo_utils.py
Helper file exists: True


### Import helper files and utility functions

In [6]:
import sys
from pathlib import Path

# Make sure Python can import helper files from the notebook folder.
sys.path.insert(0, str(Path.cwd()))

from ultralytics import YOLO
from yolo_utils import CustomizedTrainer, CustomizedValidator

### Specify the path to the files

This directory should have `train/` and `test` subdirectories inside of it.

In [ ]:
from pathlib import Path


def find_yolo_dataset_path(search_root=Path(".")):
    for folder in search_root.rglob("*"):
        if folder.is_dir() and (folder / "train").is_dir() and (folder / "test").is_dir():
            return folder.resolve()
    return None


def is_yolo_dataset_dir(path_obj: Path) -> bool:
    return path_obj.is_dir() and (path_obj / "train").is_dir() and (path_obj / "test").is_dir()


# 1) YOLO image dataset path (for this notebook's classifier workflow)
DATASET_PATH = find_yolo_dataset_path()

if DATASET_PATH is None:
    DATASET_PATH = Path("data/your_dataset")
    print("YOLO image dataset not found yet.")
    print("Create: data/your_dataset/train/<class_name>/... and data/your_dataset/test/<class_name>/...")
else:
    print(f"Using YOLO dataset: {DATASET_PATH}")

# 2) Tabular CSV dataset path (for my thesis work)
TABULAR_DATASET_PATH = Path("data/iowa_city_weather_dataset.csv")
if TABULAR_DATASET_PATH.exists():
    print(f"Found tabular CSV: {TABULAR_DATASET_PATH.resolve()}")
else:
    print("Tabular CSV not found yet. Run: python project02_weather_model.py")

YOLO image dataset not found yet.
Create: data/your_dataset/train/<class_name>/... and data/your_dataset/test/<class_name>/...
Found tabular CSV: /Users/aishakazembe/Documents/coding-cabinate/code/Intro_to_data/Intro_to_data/5005-Project02/data/iowa_city_weather_dataset.csv


### Specify the YOLO classification model

In [8]:
MODEL_NAME = "yolo11n-cls.pt"
model = YOLO(MODEL_NAME)
print(f"Loaded model: {MODEL_NAME}")

Loaded model: yolo11n-cls.pt


### Start training

This shows the training data to the model $10$ times and adjusts the model's parameters to make it more specific to our data.

In [ ]:
import torch

if is_yolo_dataset_dir(Path(DATASET_PATH)):
    if torch.cuda.is_available():
        device = 0
    elif torch.backends.mps.is_available():
        device = "mps"
    else:
        device = "cpu"

    print(f"Training on device: {device}")
    train_results = model.train(
        data=str(DATASET_PATH),
        trainer=CustomizedTrainer,
        epochs=10,
        imgsz=128,
        batch=32,
        device=device,
    )
else:
    print(f"YOLO image dataset path not ready: {DATASET_PATH}")
    print("This classifier notebook requires train/test image folders.")
    if TABULAR_DATASET_PATH.exists():
        print(f"Tabular CSV is available for separate workflow: {TABULAR_DATASET_PATH}")

Dataset path does not exist yet: data/your_dataset
Build the train/test folders first, then rerun this cell.


### Evaluate the model

Did the model learn new patterns from the new data?

In [ ]:
if is_yolo_dataset_dir(Path(DATASET_PATH)):
    val_metrics = model.val(
        data=str(DATASET_PATH),
        validator=CustomizedValidator,
        imgsz=128,
        batch=32,
        device=device,
    )

    print("Top-1 accuracy:", val_metrics.top1)
    print("Top-5 accuracy:", val_metrics.top5)
else:
    print("Skipping YOLO validation because train/test image folders are missing.")

Skipping validation because dataset path is missing.


### Look at evaluation results

There should be some files inside a `runs/classify/val` directory with information about the model's accuracy and overall performance.

## Alt project: train a weather predictive model

This section mirrors the YOLO assignment structure, but applies it to a tabular predictive-model workflow.

Goal:
- load the generated weather CSV dataset
- train baseline predictive models
- evaluate performance
- save outputs (models + metrics) in artifacts folder

### Data source acknowledgement
data used in this project comes from the Open-Meteo Historical Weather API:
- source: https://archive-api.open-meteo.com/v1/archive
- provider docs: https://open-meteo.com/
- location used in this project: Iowa City, IA (41.6611, -91.5302)

(for context, see first two projectes at (https://aishaa.net/eng-portfolio/eng-main))

the dataset file `data/iowa_city_weather_dataset.csv` is generated from that API by `project02_weather_model.py`.

### Install libraries

This mirrors the YOLO install step

In [ ]:
# Optional if this kernel is missing packages:
# %pip install -q pandas numpy scikit-learn joblib

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, f1_score
import joblib

print("tabular ML imports ready")

Tabular ML imports ready.


### Import helper files and utility functions

In [ ]:
def safe_rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def safe_f1(y_true, y_pred):
    return float(f1_score(y_true, y_pred, zero_division=0))


print("helper functions loaded.")

Helper functions loaded.


### Specify the path to the files

used the CSV generated by `project02_weather_model.py`, and create an output folder for artifacts.

artifacts created in this workflow:
- `artifacts/tabular/regression_model.joblib`
- `artifacts/tabular/classification_model.joblib`
- `artifacts/tabular/metrics_summary.csv`
- `artifacts/tabular/metrics_summary.json`

`.joblib` file?
- It is a serialized (saved) Python object file.
- each `.joblib` stores a trained scikit-learn model so you can load and reuse it later without retraining

In [5]:
TABULAR_DATASET_PATH = Path("data/iowa_city_weather_dataset.csv")
ARTIFACTS_DIR = Path("artifacts/tabular")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if TABULAR_DATASET_PATH.exists():
    df_tab = pd.read_csv(TABULAR_DATASET_PATH)
    print(f"Dataset file found: {TABULAR_DATASET_PATH.resolve()}")
    print(f"Shape: {df_tab.shape}")
else:
    print("CSV file not found.")
    print("Run: python project02_weather_model.py")

Dataset file found: /Users/aishakazembe/Documents/coding-cabinate/code/Intro_to_data/Intro_to_data/5005-Project02/data/iowa_city_weather_dataset.csv
Shape: (4134, 25)


### Specify the predictive model

picked two baseline models:
- RandomForestRegressor for next-day max temperature
- RandomForestClassifier for next-day storm prediction

In [10]:
reg_model = RandomForestRegressor(n_estimators=300, random_state=42)
cls_model = RandomForestClassifier(n_estimators=400, random_state=42, class_weight="balanced")

print("Models initialized")


Models initialized


### Start training

this trains both models and saves the fitted model files to the artifact folder.

In [11]:
if TABULAR_DATASET_PATH.exists():
    reg_features = [
        "temperature_2m_max",
        "temperature_2m_min",
        "temperature_2m_mean",
        "precipitation_sum",
        "windspeed_10m_max",
        "windgusts_10m_max",
        "temperature_2m_max_lag1",
        "temperature_2m_max_lag3",
        "temp_max_roll7",
        "month",
        "day_of_year",
    ]
    cls_features = [
        "temperature_2m_max",
        "temperature_2m_min",
        "precipitation_sum",
        "rain_sum",
        "snowfall_sum",
        "windspeed_10m_max",
        "windgusts_10m_max",
        "precipitation_sum_lag1",
        "precipitation_sum_lag3",
        "precip_roll7",
        "month",
    ]

    Xr = df_tab[reg_features]
    yr = df_tab["target_temp_max_next_day"]
    Xc = df_tab[cls_features]
    yc = df_tab["target_storm_next_day"].astype(int)

    Xr_train, Xr_test, yr_train, yr_test = train_test_split(Xr, yr, test_size=0.2, shuffle=False)
    Xc_train, Xc_test, yc_train, yc_test = train_test_split(Xc, yc, test_size=0.2, shuffle=False)

    reg_model.fit(Xr_train, yr_train)
    cls_model.fit(Xc_train, yc_train)

    reg_preds = reg_model.predict(Xr_test)
    cls_preds = cls_model.predict(Xc_test)

    joblib.dump(reg_model, ARTIFACTS_DIR / "regression_model.joblib")
    joblib.dump(cls_model, ARTIFACTS_DIR / "classification_model.joblib")

    print("Training completed, model files saved")
else:
    print("Skipping training because CSV file is missing")

Training completed, model files saved


### Evaluate the model

creates metrics and writes summary to both CSV and JSON in artifacts

In [8]:
if TABULAR_DATASET_PATH.exists():
    metrics = [
        {"model": "RandomForestRegressor", "task": "next-day max temp", "metric": "RMSE", "value": safe_rmse(yr_test, reg_preds)},
        {"model": "RandomForestRegressor", "task": "next-day max temp", "metric": "MAE", "value": float(mean_absolute_error(yr_test, reg_preds))},
        {"model": "RandomForestRegressor", "task": "next-day max temp", "metric": "R2", "value": float(r2_score(yr_test, reg_preds))},
        {"model": "RandomForestClassifier", "task": "next-day storm", "metric": "Accuracy", "value": float(accuracy_score(yc_test, cls_preds))},
        {"model": "RandomForestClassifier", "task": "next-day storm", "metric": "F1", "value": safe_f1(yc_test, cls_preds)},
    ]

    metrics_df = pd.DataFrame(metrics)
    metrics_df.to_csv(ARTIFACTS_DIR / "metrics_summary.csv", index=False)
    metrics_df.to_json(ARTIFACTS_DIR / "metrics_summary.json", orient="records", indent=2)

    print("Metrics saved to:")
    print(f"- {ARTIFACTS_DIR / 'metrics_summary.csv'}")
    print(f"- {ARTIFACTS_DIR / 'metrics_summary.json'}")
else:
    print("Skipping evaluation because CSV file is missing.")

Metrics saved to:
- artifacts/tabular/metrics_summary.csv
- artifacts/tabular/metrics_summary.json


### Look at evaluation results

The table below summarizes performance, and the file list confirms which artifacts were created.

In [9]:
if TABULAR_DATASET_PATH.exists():
    display(metrics_df)

    print("\nArtifacts in artifacts/tabular:")
    for f in sorted(ARTIFACTS_DIR.glob("*")):
        print(f"- {f.name}")
else:
    print("No results to display yet.")

,model,task,metric,value
0,RandomForestRegressor,next-day max temp,RMSE,7.203844
1,RandomForestRegressor,next-day max temp,MAE,5.557501
2,RandomForestRegressor,next-day max temp,R2,0.877070
3,RandomForestClassifier,next-day storm,Accuracy,1.000000
4,RandomForestClassifier,next-day storm,F1,0.000000



Artifacts in artifacts/tabular:
- classification_model.joblib
- metrics_summary.csv
- metrics_summary.json
- regression_model.joblib
